# On-Device Bilingual Customer Support AI

Intent Classification model (EN/JP) for mobile on-device customer support.

**Pipeline:** Train on server → export CoreML (iOS) + TFLite (Android) → deliver via platform-native asset systems.

| Setting | Value |
|---------|-------|
| Model | mmBERT-small (ModernBERT, 140M params) |
| Mode | Production (10 epochs, batch 2048, sqrt-scaled LR, A100) |
| Export | CoreML FP16 (iOS) + TFLite INT8 (Android) |

---

## Step 1: Install Dependencies

**Important:** Do NOT install `tensorflow-cpu` — Colab already has TF with GPU.
Pin `protobuf<6` to avoid conflicts with Colab's pre-installed packages.

In [ ]:
# Training
!pip install transformers>=4.51.0 accelerate -q
!pip install datasets scikit-learn openpyxl -q
!pip install matplotlib tqdm -q
!pip install gspread google-auth -q

# Export
!pip install coremltools zstandard -q
!pip install ai-edge-torch -q

# Fix protobuf — Colab's tensorflow/google-cloud need <6
!pip install "protobuf>=5.26.1,<6.0" -q

## Step 2: Clone Repository

Fill in your GitHub credentials below. The token needs `repo` scope for private repos.

In [ ]:
import os, sys

# ===========================================================
# PLATFORM DETECTION
# ===========================================================
if os.path.exists('/kaggle/working'):
    PLATFORM = 'kaggle'
elif os.path.exists('/content'):
    PLATFORM = 'colab'
else:
    PLATFORM = 'local'

print(f"Platform: {PLATFORM}")

# =============================================================
# ▼▼▼ FILL THESE IN ▼▼▼
# =============================================================
GITHUB_TOKEN = ''   # GitHub personal access token
REPO_OWNER   = 'MinhPhuPham'
REPO_NAME    = 'llm-customer-services'
REPO_BRANCH  = 'main'
# =============================================================

try:
    if PLATFORM == 'kaggle':
        from kaggle_secrets import UserSecretsClient
        GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN_CUSTOMER_SERVICE') or ''
    elif PLATFORM == 'colab':
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN_CUSTOMER_SERVICE') or ''
except Exception:
    pass

# Build repo URL (private with token, public without)
if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('Using private repo (token set)')
else:
    REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('Using public repo (no token)')

# Repo directory
if os.path.exists('/content'):
    REPO_DIR = f'/content/{REPO_NAME}'
elif os.path.exists('/kaggle/working'):
    REPO_DIR = f'/kaggle/working/{REPO_NAME}'
else:
    REPO_DIR = os.path.expanduser(f'~/{REPO_NAME}')

# Clone or pull
if not os.path.exists(REPO_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {REPO_BRANCH} && git pull origin {REPO_BRANCH}

# Add to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'\nRepo dir: {REPO_DIR}')

## Step 3: Mount Google Drive & Setup Directories

In [ ]:
# Mount Drive (Colab only)
if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')

# Create output directories
from scripts.setup import create_directories
create_directories()

## Step 4: Configuration

Default: **Production mode** — 10 epochs, batch 2048, sqrt-scaled LR (1.6e-4) on A100.

In [ ]:
from scripts.config import print_config
print_config()

# Override if needed:
# import scripts.config as cfg
# cfg.TESTING_MODE = True   # Quick test: 3 epochs, batch 32
# cfg.NUM_EPOCHS = 10
# cfg.BATCH_SIZE = 64

## Step 5: Load Training Data

Choose **one** of the two data sources below:
- **Option A:** Excel file (.xlsx) on Drive or local path
- **Option B:** Google Sheets (directly from a Sheet URL)

> **Note:** If your Excel file on Drive is opened/edited via Google Sheets,
> Drive keeps it as `.xlsx` — Option A works fine.
> If you converted it to a native Google Sheet, use Option B.

In [ ]:
from scripts.config import DATA_DIR, BASE_MODEL, EXPORT_DIR
from scripts.data_loader.excel_parser import parse_excel, parse_google_sheet
from scripts.data_loader.dataset_builder import prepare_splits, tokenize_datasets
from scripts.data_loader.export_artifacts import export_label_map, export_responses
from transformers import AutoTokenizer

# =============================================================
# OPTION A: Excel file (default — uncomment the path you use)
# =============================================================
# excel_candidates = [
#     os.path.join(DATA_DIR, 'training_data_template.xlsx'),
#     os.path.join(REPO_DIR, 'service_model_training_data_template.xlsx'),
# ]
# excel_path = next((p for p in excel_candidates if os.path.exists(p)), None)

# if excel_path:
#     train_rows, df = parse_excel(excel_path)
# else:
#     print('Excel file not found. Trying Google Sheets (Option B)...')
#     train_rows, df = None, None

# =============================================================
# OPTION B: Google Sheets (uncomment below to use)
# =============================================================
# First, authenticate with Google:
from google.colab import auth
auth.authenticate_user()

# Then load by URL:
train_rows, df = parse_google_sheet(
    sheet_url='https://docs.google.com/spreadsheets/d/1nVFgUw5O2z6YsRbITxGRY9VXNHyLTaGh4VFKS5P77pg/edit?gid=1475296463#gid=1475296463'
)

# Or load by sheet name:
# train_rows, df = parse_google_sheet(sheet_name='training_data_template')

assert train_rows is not None, 'No data loaded. Check Excel path or enable Google Sheets option.'

In [ ]:
# Prepare label encoding + train/val splits (NO tokenization yet —
# tokenization happens after vocab pruning so IDs match the pruned embeddings)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

(
    texts, train_texts, val_texts,
    train_labels, val_labels,
    label_encoder, num_labels,
) = prepare_splits(train_rows)

label_map = export_label_map(label_encoder)
responses = export_responses(df)

## Step 6: Vocabulary Pruning + Tokenization

Strip unused language tokens from the embedding matrix (~256K → ~35K), then
tokenize with a remapped tokenizer so token IDs match the pruned embeddings.

In [ ]:
from scripts.helpers.vocab_pruner import prune_vocabulary, RemappedTokenizer

base_model, kept_ids, old_to_new = prune_vocabulary(tokenizer, texts)

# Wrap tokenizer so IDs map to the pruned embedding matrix
tokenizer = RemappedTokenizer(tokenizer, old_to_new)

# NOW tokenize with remapped IDs
train_ds, val_ds = tokenize_datasets(
    train_texts, val_texts, train_labels, val_labels, tokenizer
)

## Step 7: Fine-tune

Production: 10 epochs, batch 2048, cosine LR (1.6e-4, sqrt-scaled) on A100.

In [ ]:
from scripts.helpers.trainer_factory import build_trainer, run_training

trainer, classifier = build_trainer(
    base_model, kept_ids, num_labels, train_ds, val_ds, tokenizer
)
history = run_training(trainer)

## Step 8: Training Plots

In [ ]:
from scripts.helpers.training_plots import plot_training_history

best_accuracy = plot_training_history(trainer, show=True)

## Step 9: Export CoreML (iOS)

In [ ]:
from scripts.helpers.export_coreml import export_coreml

coreml_path, coreml_size = export_coreml(trainer, tokenizer)

## Step 10: Export TFLite (Android)

In [ ]:
from scripts.helpers.export_tflite import export_tflite

tflite_path, tflite_size = export_tflite(
    tokenizer=tokenizer,
    calibration_texts=val_texts,
)

## Step 11: Compress for OTA Delivery

In [ ]:
from scripts.helpers.compress import compress_for_ota

compress_for_ota([('TFLite', tflite_path)])

## Step 12: Evaluate

In [ ]:
from scripts.helpers.evaluator import evaluate_model

results = evaluate_model(
    tflite_path, tokenizer, label_map, label_encoder,
    val_texts, val_labels,
)

## Step 13: Export Summary

In [ ]:
from scripts.config import EXPORT_DIR

export_files = {
    'CoreML (iOS)': os.path.join(EXPORT_DIR, 'SupportAI.mlpackage'),
    'TFLite (Android)': os.path.join(EXPORT_DIR, 'support_ai.tflite'),
    'TFLite compressed': os.path.join(EXPORT_DIR, 'support_ai.tflite.zst'),
    'label_map.json': os.path.join(EXPORT_DIR, 'label_map.json'),
    'responses.json': os.path.join(EXPORT_DIR, 'responses.json'),
    'token_id_map.json': os.path.join(EXPORT_DIR, 'token_id_map.json'),
    'training_plots.png': os.path.join(EXPORT_DIR, 'training_plots.png'),
}

total = 0
for name, path in export_files.items():
    if os.path.isdir(path):
        sz = sum(os.path.getsize(os.path.join(dp, f))
                 for dp, _, fn in os.walk(path) for f in fn) / 1e6
    elif os.path.exists(path):
        sz = os.path.getsize(path) / 1e6
    else:
        sz = 0
    total += sz
    print(f'{name:25s} {sz:8.1f}MB')

print(f'{"TOTAL":25s} {total:8.1f}MB')
print(f'\nSaved to: {EXPORT_DIR}')

---

## ✅ Done!

Export artifacts are saved to Google Drive / output directory.

**Next steps:**
- Copy `SupportAI.mlpackage` → Xcode project (iOS)
- Copy `support_ai.tflite` → Android assets (or Play Asset Delivery)
- Bundle `responses.json` + `label_map.json` in both apps
- For OTA: upload `.zst` files to CDN with `manifest.json`